# EasyAgent 内置工具测试

每个 Cell 独立测试一种内置工具，直接调用 `tool.run()` 验证功能正确性，不依赖 LLM。

**覆盖工具清单 (27 个)**：
Calculator, FileRead, Glob, Grep, FileWrite, FileEdit, Bash, TodoWrite,
TaskCreate/Get/Update/List, Config, WebFetch, WebSearch, NotebookEdit,
TaskOutput, TaskStop, Agent, SendMessage, TeamCreate, TeamDelete,
AskUserQuestion, EnterPlanMode, ExitPlanMode, EnterWorktree, ExitWorktree

> MCP 工具（MCPWrappedTool 等）需要外部 MCP Server，不在此测试。

In [ ]:
# ==================== 环境初始化 ====================
import os, sys, json, shutil
from pathlib import Path
from dotenv import load_dotenv

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

load_dotenv(os.path.join(os.getcwd(), ".env"))

from Tool.ToolRegistry import ToolRegistry
from Tool.BaseTool import ToolResult

SCRATCH_DIR = os.path.join(os.getcwd(), "scratch", "tool_test")
os.makedirs(SCRATCH_DIR, exist_ok=True)

def show_result(result):
    if isinstance(result, ToolResult):
        print(f"状态: {result.status}")
        print(f"内容: {result.to_display_string()[:500]}")
        if result.structured_data:
            print(f"结构化数据: {json.dumps(result.structured_data, ensure_ascii=False, indent=2, default=str)[:500]}")
        if result.error_type:
            print(f"错误类型: {result.error_type}")
    else:
        print(f"原始返回: {str(result)[:500]}")
    return result

print(f"项目根目录: {project_root}")
print(f"临时测试目录: {SCRATCH_DIR}")
print("✅ 环境初始化完成")

# [Agent 调用测试]
from agent import BasicAgent
from core.llm import EasyLLM

llm = EasyLLM(
    provider="openai",
    base_url="http://127.0.0.1:5124/v1",
    api_key="122",
    model="qwen3.5-9b",
)

async def test_agent_with_tool(registry, prompt: str):
    agent = BasicAgent(
        name="TestAgent",
        llm=llm,
        tool_registry=registry,
        enable_tool=True,
    )
    print("\n=== [Agent 调用测试] ===")
    print(f"提问: {prompt}")
    try:
        res = await agent.astream_invoke(prompt, max_iter=3)
        print(f"Agent最终回复: {res}")
    except Exception as e:
        print(f"Agent调用异常: {e}")


In [ ]:
# ==================== 1. Calculator Tool ====================
from Tool.builtin import register_calculator_tool

registry = ToolRegistry()
calc_tool = register_calculator_tool(registry)

print("=== 基本四则运算 ===")
show_result(calc_tool.run({"expression": "2 + 3 * 4"}))

print("\n=== 数学函数 ===")
show_result(calc_tool.run({"expression": "sqrt(16) + pow(2, 3)"}))

print("\n=== 常量 ===")
show_result(calc_tool.run({"expression": "sin(pi / 2)"}))

print("\n=== 中文符号 ===")
show_result(calc_tool.run({"expression": "（3＋2）×10"}))

print("\n=== 错误：除以零 ===")
show_result(calc_tool.run({"expression": "1 / 0"}))

print("\n=== 错误：空表达式 ===")
show_result(calc_tool.run({"expression": ""}))

print("\n✅ Calculator Tool 测试完成")

await test_agent_with_tool(registry, '请帮我计算一下 (3+2)*10 等于多少？')


In [ ]:
# ==================== 2. FileRead Tool ====================
from Tool.builtin import register_file_read_tool

registry = ToolRegistry()
read_tool = register_file_read_tool(registry, workspace_root=project_root)

test_file = os.path.join(SCRATCH_DIR, "read_test.txt")
with open(test_file, "w") as f:
    for i in range(20):
        f.write(f"Line {i+1}: This is test content\n")

print("=== 读取完整文件 ===")
show_result(read_tool.run({"file_path": test_file}))

print("\n=== 带 offset 和 limit ===")
show_result(read_tool.run({"file_path": test_file, "offset": 5, "limit": 3}))

print("\n=== 读取不存在的文件 ===")
show_result(read_tool.run({"file_path": "/nonexistent/file.txt"}))

print("\n✅ FileRead Tool 测试完成")

await test_agent_with_tool(registry, f'请读取当前目录下的临时文件 {test_file} 的前三行并告诉我内容。')


In [ ]:
# ==================== 3. Glob Tool ====================
from Tool.builtin import register_glob_tool

registry = ToolRegistry()
glob_tool = register_glob_tool(registry, workspace_root=project_root)

for name in ["a.py", "b.py", "c.txt", "sub/d.py"]:
    path = os.path.join(SCRATCH_DIR, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    Path(path).touch()

print("=== 搜索 SCRATCH_DIR 下的 .py 文件 ===")
show_result(glob_tool.run({"pattern": "**/*.py", "path": SCRATCH_DIR}))

print("\n=== 搜索 .txt 文件 ===")
show_result(glob_tool.run({"pattern": "*.txt", "path": SCRATCH_DIR}))

print("\n✅ Glob Tool 测试完成")

await test_agent_with_tool(registry, f'请搜索 {SCRATCH_DIR} 目录下所有的 .py 文件有哪些？')


In [ ]:
# ==================== 4. Grep Tool ====================
from Tool.builtin import register_grep_tool

registry = ToolRegistry()
grep_tool = register_grep_tool(registry, workspace_root=project_root)

grep_file = os.path.join(SCRATCH_DIR, "grep_test.py")
with open(grep_file, "w") as f:
    f.write("def hello():\n    print('Hello World')\n\ndef goodbye():\n    print('Goodbye World')\n")

print("=== 搜索包含 'Hello' 的行 ===")
show_result(grep_tool.run({"pattern": "Hello", "path": SCRATCH_DIR, "output_mode": "content"}))

print("\n=== 搜索包含 'def' 的文件 ===")
show_result(grep_tool.run({"pattern": "def", "path": SCRATCH_DIR, "output_mode": "files_with_matches"}))

print("\n=== 搜索不存在的模式 ===")
show_result(grep_tool.run({"pattern": "NONEXISTENT_PATTERN_XYZ", "path": SCRATCH_DIR}))

print("\n✅ Grep Tool 测试完成")

await test_agent_with_tool(registry, f'请帮我在 {SCRATCH_DIR} 下搜索包含 "Hello" 的内容。')


In [ ]:
# ==================== 5. FileWrite Tool ====================
from Tool.builtin import register_file_write_tool

registry = ToolRegistry()
write_tool = register_file_write_tool(registry, workspace_root=SCRATCH_DIR)

write_path = os.path.join(SCRATCH_DIR, "write_test.txt")

print("=== 写入新文件 ===")
show_result(write_tool.run({"file_path": write_path, "content": "Hello from FileWrite Tool!\nLine 2\nLine 3"}))

print("\n=== 验证文件内容 ===")
with open(write_path, "r") as f:
    content = f.read()
print(f"文件内容:\n{content}")
assert "Hello from FileWrite Tool!" in content, "写入内容不匹配！"

print("\n=== 覆盖写入 ===")
show_result(write_tool.run({"file_path": write_path, "content": "Overwritten content"}))
with open(write_path, "r") as f:
    assert f.read().strip() == "Overwritten content", "覆盖写入失败！"

print("\n✅ FileWrite Tool 测试完成")

await test_agent_with_tool(registry, f'请把一段测试文本 "Hello from agent!" 写入到 {write_path} 文件中，完全覆盖写入。')


In [ ]:
# ==================== 6. FileEdit Tool ====================
from Tool.builtin import register_file_edit_tool

registry = ToolRegistry()
edit_tool = register_file_edit_tool(registry, workspace_root=SCRATCH_DIR)
read_tool = register_file_read_tool(registry,workspace_root=SCRATCH_DIR)
edit_path = os.path.join(SCRATCH_DIR, "edit_test.py")
with open(edit_path, "w") as f:
    f.write("def add(a, b):\n    return a + b\n")
print("=== 编辑前内容 ===")
show_result(
    read_tool.run({"file_path": edit_path})

)

print("=== 执行编辑：添加类型注解 ===")

show_result(edit_tool.run({
    "file_path": edit_path,
    "old_string": "def add(a, b):",
    "new_string": "def add(a: int, b: int) -> int:",
}))

print("\n=== 编辑后内容 ===")
with open(edit_path, "r") as f:
    content = f.read()
print(content)
assert "a: int" in content, "编辑未生效！"

print("\n=== 错误：old_string 不存在 ===")
show_result(edit_tool.run({
    "file_path": edit_path,
    "old_string": "THIS_DOES_NOT_EXIST",
    "new_string": "REPLACEMENT",
}))

print("\n✅ FileEdit Tool 测试完成")

await test_agent_with_tool(registry, f'请把文件 {edit_path} 中的 "def add(a: int, b: int) -> int:" 替换成没有类型注解的 "def add(a, b):"。')


In [ ]:
# ==================== 7. Bash Tool ====================
from Tool.builtin import register_shell_tools

registry = ToolRegistry()
register_shell_tools(registry, workspace_root=project_root)
bash_tool = registry.get_tool("Bash")

print("=== 运行 echo 命令 ===")
show_result(bash_tool.run({"command": "echo 'Hello from Bash Tool!'"}))

print("\n=== 运行 ls 命令 ===")
show_result(bash_tool.run({"command": f"ls {SCRATCH_DIR}"}))

print("\n=== 运行 python -c ===")
show_result(bash_tool.run({"command": "python3 -c 'print(2 + 2)'"}))

print("\n=== 错误：不存在的命令 ===")
show_result(bash_tool.run({"command": "nonexistent_command_xyz 2>&1 || true"}))

print("\n✅ Bash Tool 测试完成")

await test_agent_with_tool(registry, '请通过终端帮我执行一个 echo 命令输出 Hello bash agent!。')


In [ ]:
# ==================== 8. TodoWrite Tool ====================
from Tool.builtin import register_todo_write_tool
from Tool.runtime import clear_todo_items

clear_todo_items()

registry = ToolRegistry()
todo_tool = register_todo_write_tool(registry)

print("=== 创建初始 TODO 列表 ===")
show_result(todo_tool.run({
    "todos": [
        {"content": "重构数据库连接池", "status": "pending", "activeForm": "准备重构数据库连接池"},
        {"content": "编写单元测试", "status": "in_progress", "activeForm": "正在编写单元测试"},
        {"content": "更新文档", "status": "completed", "activeForm": "正在更新文档"},
    ]
}))

print("\n=== 更新 TODO 列表 ===")
show_result(todo_tool.run({
    "todos": [
        {"content": "重构数据库连接池", "status": "in_progress", "activeForm": "正在重构数据库连接池"},
        {"content": "编写单元测试", "status": "completed", "activeForm": "已完成单元测试"},
        {"content": "更新文档", "status": "completed", "activeForm": "已更新文档"},
        {"content": "验证回归测试", "status": "pending", "activeForm": "准备验证回归测试"},
    ]
}))

print("\n=== 错误：重复 content ===")
show_result(todo_tool.run({
    "todos": [
        {"content": "任务A", "status": "pending", "activeForm": "准备做A"},
        {"content": "任务A", "status": "pending", "activeForm": "准备做A"},
    ]
}))

clear_todo_items()
print("\n✅ TodoWrite Tool 测试完成")

await test_agent_with_tool(registry, '当前有一个重构数据库连池的TODO，请帮我把状态改为已完成。')


In [ ]:
# ==================== 9. TaskCreate / TaskGet / TaskUpdate / TaskList ====================
from Tool.builtin import register_task_tools
from task import SQLiteTaskStore, TaskService

db_path = os.path.join(SCRATCH_DIR, "test_tasks.db")
if os.path.exists(db_path):
    os.remove(db_path)
service = TaskService(SQLiteTaskStore(db_path))

registry = ToolRegistry()
create_tool, get_tool, update_tool, list_tool = register_task_tools(registry, service=service)

print("=== TaskCreate: 创建任务 ===")
r1 = create_tool.run({"title": "实现用户认证", "description": "添加 JWT 认证", "status": "open"})
show_result(r1)
task_id_1 = r1.metadata.get("task_id")

print("\n=== TaskCreate: 创建第二个任务 ===")
r2 = create_tool.run({"title": "编写 API 文档", "status": "open", "owner": "dev-team"})
show_result(r2)
task_id_2 = r2.metadata.get("task_id")

print("\n=== TaskGet: 查询任务 ===")
show_result(get_tool.run({"task_id": task_id_1}))

print("\n=== TaskUpdate: 更新任务状态 ===")
show_result(update_tool.run({"task_id": task_id_1, "status": "in_progress"}))

print("\n=== TaskList: 列出所有任务 ===")
show_result(list_tool.run({}))

print("\n=== TaskList: 按状态过滤 ===")
show_result(list_tool.run({"status": "open"}))

print("\n=== TaskGet: 查询不存在的任务 ===")
show_result(get_tool.run({"task_id": "nonexistent-id"}))

# os.remove(db_path)
print("\n✅ Task Tools 测试完成")

await test_agent_with_tool(registry, '请帮我查一下现在有哪些任务？如果有任务，请帮我把状态改为 completed。')


In [ ]:
# ==================== 10. Config Tool ====================
from Tool.builtin import register_config_tool
from core.Config import Config

config = Config(workspace_root=project_root, allowed_roots=[project_root])
registry = ToolRegistry()
config_tool = register_config_tool(registry, config=config)

print("=== 读取配置项 ===")
show_result(config_tool.run({"setting": "workspace_root"}))

print("\n=== 读取另一个配置项 ===")
show_result(config_tool.run({"setting": "max_background_tasks"}))

print("\n=== 更新配置项 ===")
show_result(config_tool.run({"setting": "max_background_tasks", "value": "16"}))

print("\n=== 验证更新 ===")
show_result(config_tool.run({"setting": "max_background_tasks"}))

print("\n=== 错误：不存在的配置项 ===")
show_result(config_tool.run({"setting": "nonexistent_key"}))

print("\n=== 错误：空 setting ===")
show_result(config_tool.run({"setting": ""}))

print("\n✅ Config Tool 测试完成")

await test_agent_with_tool(registry, '请帮我查一下当前的 workspace_root 配置是什么？然后尝试将其改为新的路径。')


In [ ]:
# ==================== 11. WebFetch Tool ====================
from Tool.builtin import register_web_fetch_tool

registry = ToolRegistry()
fetch_tool = register_web_fetch_tool(registry)

print("=== 抓取 example.com ===")
show_result(fetch_tool.run({"url": "https://example.com", "prompt": "提取页面标题和主要内容"}))

print("\n=== 错误：空 URL ===")
show_result(fetch_tool.run({"url": "", "prompt": "test"}))

print("\n=== 错误：无效 URL ===")
show_result(fetch_tool.run({"url": "ftp://invalid", "prompt": "test"}))

print("\n=== 错误：空 prompt ===")
show_result(fetch_tool.run({"url": "https://example.com", "prompt": ""}))

print("\n✅ WebFetch Tool 测试完成")

await test_agent_with_tool(registry, '请帮我抓取 https://example.com 并提取一下主要内容。')


In [ ]:
# ==================== 12. WebSearch Tool ====================
from Tool.builtin import register_search_tool

registry = ToolRegistry()
search_tool = register_search_tool(registry)

print(f"搜索后端: {search_tool.backend}")

print("\n=== 搜索 Python 教程 ===")
show_result(search_tool.run({"query": "Python asyncio tutorial", "num_results": 3}))

print("\n=== 错误：空查询 ===")
show_result(search_tool.run({"query": ""}))

print("\n✅ WebSearch Tool 测试完成")

await test_agent_with_tool(registry, '请帮我搜索一下 python asyncio 的最新教程。')


In [9]:
# ==================== 13. NotebookEdit Tool ====================
from Tool.builtin import register_notebook_edit_tool,register_file_read_tool

registry = ToolRegistry()
nb_tool = register_notebook_edit_tool(registry, workspace_root=SCRATCH_DIR)

nb_path = os.path.join(SCRATCH_DIR, "test_notebook.ipynb")
nb_content = {
    "cells": [],
    "metadata": {"kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"}},
    "nbformat": 4,
    "nbformat_minor": 5
}
with open(nb_path, "w") as f:
    json.dump(nb_content, f)
read_tool = register_file_read_tool(registry,workspace_root=SCRATCH_DIR)

read_tool.run({"file_path": nb_path})
print("=== 插入一个代码 cell ===")
show_result(nb_tool.run({
    "notebook_path": nb_path,
    "new_source": "print('Hello from notebook!')",
    "cell_type": "code",
    "edit_mode": "insert",
}))

print("\n=== 验证 notebook 内容 ===")
with open(nb_path, "r") as f:
    nb = json.load(f)
print(f"Cell 数量: {len(nb['cells'])}")
if nb["cells"]:
    print(f"第一个 cell 类型: {nb['cells'][0].get('cell_type')}")
    print(f"第一个 cell 内容: {nb['cells'][0].get('source')}")

print("\n✅ NotebookEdit Tool 测试完成")

await test_agent_with_tool(registry, f'请在 notebook {nb_path} 中插入一个包含 print(Hello) 的代码块。')


=== 插入一个代码 cell ===
状态: success
内容: 已更新 Notebook: /home/wxd/LLM/EasyAgent/example/scratch/tool_test/test_notebook.ipynb
结构化数据: {
  "notebookPath": "/home/wxd/LLM/EasyAgent/example/scratch/tool_test/test_notebook.ipynb",
  "editMode": "insert",
  "cellId": null,
  "affectedCellId": "bf237c10",
  "cellIndex": 0,
  "oldCell": null,
  "newCell": {
    "cell_type": "code",
    "id": "bf237c10",
    "metadata": {},
    "source": [
      "print('Hello from notebook!')"
    ],
    "execution_count": null,
    "outputs": []
  },
  "fileVersion": {
    "path": "/home/wxd/LLM/EasyAgent/example/scratch/tool_test/test_notebook.ipynb

=== 验证 notebook 内容 ===
Cell 数量: 1
第一个 cell 类型: code
第一个 cell 内容: ["print('Hello from notebook!')"]

✅ NotebookEdit Tool 测试完成

=== [Agent 调用测试] ===
提问: 请在 notebook /home/wxd/LLM/EasyAgent/example/scratch/tool_test/test_notebook.ipynb 中插入一个包含 print(Hello) 的代码块。
round 1

thinking content:
用户想在 notebook 中插入一个包含 print(Hello) 的代码块。我需要：

1. 先读取这个 notebook 文件，了解它的结构
2. 然后使用 No

In [11]:
# ==================== 14. TaskOutput + TaskStop Tools ====================
from Tool.builtin import register_shell_tools

registry = ToolRegistry()
register_shell_tools(registry, workspace_root=project_root)
bash_tool = registry.get_tool("Bash")
output_tool = registry.get_tool("TaskOutput")
stop_tool = registry.get_tool("TaskStop")

print("=== 启动后台任务 ===")
bg_result = bash_tool.run({"command": "sleep 2 && ls ", "run_in_background": True})
show_result(bg_result)
task_id = bg_result.metadata.get("task_id", "") if isinstance(bg_result, ToolResult) else ""
print(f"后台任务 ID: {task_id}")

if task_id:
    print("\n=== TaskOutput: 读取后台任务输出 ===")
    show_result(output_tool.run({"task_id": task_id, "block": True, "timeout": 5000}))

print("\n=== TaskOutput: 查询不存在的任务 ===")
show_result(output_tool.run({"task_id": "nonexistent-task"}))

print("\n=== TaskStop: 停止不存在的任务 ===")
show_result(stop_tool.run({"task_id": "nonexistent-task"}))

print("\n✅ TaskOutput & TaskStop Tools 测试完成")

if task_id:
    await test_agent_with_tool(registry, f'请帮我查询后台任务 {task_id} 的输出。')


=== 启动后台任务 ===
状态: success
内容: 任务 ID: task_17ad8d17bbe6

状态: running

命令: sleep 2 && ls

cwd: /home/wxd/LLM/EasyAgent
结构化数据: {
  "task_id": "task_17ad8d17bbe6",
  "command": "sleep 2 && ls",
  "cwd": "/home/wxd/LLM/EasyAgent",
  "status": "running",
  "return_code": null,
  "stdout": "",
  "stderr": "",
  "started_at": 1776615737.7399397,
  "finished_at": null,
  "description": null,
  "truncated": false
}
后台任务 ID: task_17ad8d17bbe6

=== TaskOutput: 读取后台任务输出 ===
状态: success
内容: 任务 ID: task_17ad8d17bbe6

状态: completed

命令: sleep 2 && ls

cwd: /home/wxd/LLM/EasyAgent

退出码: 0

stdout:
agent
conftest.py
context
core
db
docs
example
fix_md.py
__init__.py
mcp
memory
orchestrator
output
prompt
__pycache__
pytest.ini
rag
README.md
requirements
requirements.txt
runtime
scratch
scratch.py
skill
task
test
test_openai.py
Tool

结构化数据: {
  "task_id": "task_17ad8d17bbe6",
  "command": "sleep 2 && ls",
  "cwd": "/home/wxd/LLM/EasyAgent",
  "status": "completed",
  "return_code": 0,
  "stdout": "agent\

In [12]:
# ==================== 15. Agent Tool ====================
from Tool.builtin import register_agent_tool, register_filesystem_tools
from core.llm import EasyLLM
from agent import BasicAgent

llm = EasyLLM(provider="openai")
registry = ToolRegistry()
register_filesystem_tools(registry, workspace_root=project_root)

parent_agent = BasicAgent(
    name="ParentAgent",
    llm=llm,
    tool_registry=registry,
    enable_tool=True,
)

agent_tool = register_agent_tool(
    registry,
    parent_agent=parent_agent,
    workspace_root=project_root,
    allowed_roots=(project_root,),
    storage_dir=os.path.join(SCRATCH_DIR, ".agents"),
    max_background_tasks=2,
)

print("=== AgentTool: Schema 检查 ===")
schema = agent_tool.get_openai_schema()
print(f"工具名: {schema['function']['name']}")
print(f"参数 properties: {list(schema['function']['parameters'].get('properties', {}).keys())}")
print(f"参数 required: {schema['function']['parameters'].get('required', [])}")

print("\n=== AgentTool: 缺少 description ===")
show_result(agent_tool.run({"description": "", "prompt": "test"}))

print("\n=== AgentTool: 缺少 prompt ===")
show_result(agent_tool.run({"description": "test task", "prompt": ""}))

print("\n=== AgentTool: 前台启动子 agent (简单任务，需要 LLM 可用) ===")
result = agent_tool.run({
    "description": "列出项目文件",
    "prompt": "请使用 Glob 工具列出当前项目根目录下的 *.md 文件，然后直接输出结果。",
    "name": "file-lister",
})
show_result(result)

print("\n✅ Agent Tool 测试完成")

await test_agent_with_tool(registry, '请帮我启动一个名为 helper 的子 agent，让它后台执行去查一下项目根目录下有哪些文件。')


=== AgentTool: Schema 检查 ===
工具名: Agent
参数 properties: ['description', 'prompt', 'subagent_type', 'model', 'run_in_background', 'name', 'team_name', 'mode', 'isolation']
参数 required: ['description', 'prompt']

=== AgentTool: 缺少 description ===
状态: error
内容: 启动子 agent 失败: description 不能为空。
错误类型: invalid_parameters

=== AgentTool: 缺少 prompt ===
状态: error
内容: 启动子 agent 失败: prompt 不能为空。
错误类型: invalid_parameters

=== AgentTool: 前台启动子 agent (简单任务，需要 LLM 可用) ===
状态: success
内容: 当前项目根目录下的 `.md` 文件如下：

- `README.md`
结构化数据: {
  "agentId": "agent_2232c8bd883b",
  "status": "completed",
  "description": "列出项目文件",
  "prompt": "请使用 Glob 工具列出当前项目根目录下的 *.md 文件，然后直接输出结果。",
  "outputFile": "/home/wxd/LLM/EasyAgent/example/scratch/tool_test/.agents/agent_2232c8bd883b.md",
  "workspaceRoot": "/home/wxd/LLM/EasyAgent",
  "allowedRoots": [
    "/home/wxd/LLM/EasyAgent"
  ],
  "executionContext": {
    "workspaceRoot": "/home/wxd/LLM/EasyAgent",
    "allowedRoots": [
      "/home/wxd/LLM/EasyAgent"
    ],
  

In [ ]:
# ==================== 16. SendMessage Tool ====================
from Tool.builtin import register_send_message_tool, register_agent_tool
from runtime import TeamManager
from core.llm import EasyLLM
from agent import BasicAgent

llm = EasyLLM(provider="openai")
registry = ToolRegistry()

agent = BasicAgent(name="TestSender", llm=llm, tool_registry=registry, enable_tool=True)
agent_tool = register_agent_tool(registry, parent_agent=agent, workspace_root=project_root)
team_manager = TeamManager(agent_runtime=agent_tool.agent_runtime)
agent_tool.agent_runtime.bind_team_manager(team_manager)

msg_tool = register_send_message_tool(registry, agent_runtime=agent_tool.agent_runtime, parent_agent=agent)

print("=== 发送消息到不存在的 agent ===")
show_result(msg_tool.run({
    "recipient_type": "agent",
    "recipient_id": "nonexistent-agent",
    "content": "Hello!",
}))

print("\n✅ SendMessage Tool 测试完成")

await test_agent_with_tool(registry, '请将一句问候语偷偷发给名为 somebody 的 agent。')


In [ ]:
# ==================== 17. TeamCreate + TeamDelete Tools ====================
from Tool.builtin import register_team_create_tool, register_team_delete_tool, register_agent_tool
from runtime import TeamManager
from core.llm import EasyLLM
from agent import BasicAgent

llm = EasyLLM(provider="openai")
registry = ToolRegistry()
agent = BasicAgent(name="TeamTester", llm=llm, tool_registry=registry, enable_tool=True)
agent_tool = register_agent_tool(registry, parent_agent=agent, workspace_root=project_root)
team_manager = TeamManager(agent_runtime=agent_tool.agent_runtime)
agent_tool.agent_runtime.bind_team_manager(team_manager)

tc_tool = register_team_create_tool(registry, team_manager=team_manager)
td_tool = register_team_delete_tool(registry, team_manager=team_manager)

print("=== TeamCreate: 创建团队 ===")
tc_result = tc_tool.run({"name": "test-team", "description": "测试团队"})
show_result(tc_result)
team_id = tc_result.structured_data.get("team_id", "") if isinstance(tc_result, ToolResult) and tc_result.structured_data else ""
print(f"团队 ID: {team_id}")

print("\n=== TeamCreate: 重复创建同名团队 ===")
show_result(tc_tool.run({"name": "test-team", "description": "重复"}))

if team_id:
    print(f"\n=== TeamDelete: 删除团队 {team_id} ===")
    show_result(td_tool.run({"team_id": team_id}))

print("\n=== TeamDelete: 删除不存在的团队 ===")
show_result(td_tool.run({"team_id": "nonexistent"}))

print("\n✅ TeamCreate & TeamDelete Tools 测试完成")

await test_agent_with_tool(registry, '请创建一个叫 new-team 的团队，描述随意。然后查询一下是不是创建成功了？')


In [13]:
# ==================== 18. AskUserQuestion Tool ====================
from Tool.builtin import register_ask_user_question_tool

registry = ToolRegistry()
ask_tool = register_ask_user_question_tool(registry)

print("=== 结构化提问 ===")
result = ask_tool.run({
    "questions": [
        {
            "question": "你希望使用哪种数据库？",
            "header": "数据库选择",
            "options": [
                {"label": "PostgreSQL", "description": "关系型数据库，适合复杂查询"},
                {"label": "MongoDB", "description": "文档型数据库，适合灵活 Schema"},
            ],
        }
    ],
    "source": "test",
})
show_result(result)
assert result.status == "needs_confirmation", "AskUserQuestion 应返回 needs_confirmation"

print("\n=== 多问题提问 ===")
result2 = ask_tool.run({
    "questions": [
        {
            "question": "框架选择？",
            "header": "框架",
            "options": [
                {"label": "FastAPI", "description": "异步"},
                {"label": "Flask", "description": "轻量"},
            ],
        },
        {
            "question": "部署方式？",
            "header": "部署",
            "options": [
                {"label": "Docker", "description": "容器化"},
                {"label": "直接部署", "description": "原生"},
            ],
        },
    ],
})
show_result(result2)
assert result2.structured_data["questions"].__len__() == 2

print("\n✅ AskUserQuestion Tool 测试完成")

await test_agent_with_tool(registry, '请向我提问，问我喜欢 python 还是 java，并提供相应的选项让我选。')


=== 结构化提问 ===
状态: needs_confirmation
内容: 需要用户回答 1 个结构化问题后才能继续执行。
结构化数据: {
  "questions": [
    {
      "question": "你希望使用哪种数据库？",
      "header": "数据库选择",
      "options": [
        {
          "label": "PostgreSQL",
          "description": "关系型数据库，适合复杂查询"
        },
        {
          "label": "MongoDB",
          "description": "文档型数据库，适合灵活 Schema"
        }
      ]
    }
  ],
  "source": "test",
  "message": "需要用户回答 1 个结构化问题后才能继续执行。"
}
错误类型: ask_user_question

=== 多问题提问 ===
状态: needs_confirmation
内容: 需要用户回答 2 个结构化问题后才能继续执行。
结构化数据: {
  "questions": [
    {
      "question": "框架选择？",
      "header": "框架",
      "options": [
        {
          "label": "FastAPI",
          "description": "异步"
        },
        {
          "label": "Flask",
          "description": "轻量"
        }
      ]
    },
    {
      "question": "部署方式？",
      "header": "部署",
      "options": [
        {
          "label": "Docker",
          "description": "容器化"
        },
        {
          "label": "直接部署",

In [14]:
# ==================== 19. EnterPlanMode + ExitPlanMode Tools ====================
from Tool.builtin import register_enter_plan_mode_tool, register_exit_plan_mode_tool

registry = ToolRegistry()
enter_plan_tool = register_enter_plan_mode_tool(registry)
exit_plan_tool = register_exit_plan_mode_tool(registry)

print("=== EnterPlanMode: 请求进入计划模式 ===")
result = enter_plan_tool.run({
    "reason": "需求尚不明确，需要先做技术调研",
    "allowedActions": ["read_files", "search"],
})
show_result(result)
assert result.status == "needs_confirmation", "EnterPlanMode 应返回 needs_confirmation"
assert result.error_type == "enter_plan_mode_requested"
print(f"交互类型: {result.metadata.get('interaction_type')}")

print("\n=== ExitPlanMode: 请求退出计划模式 ===")
result2 = exit_plan_tool.run({
    "allowedPrompts": [
        {"tool": "Bash", "prompt": "允许执行构建命令"},
    ],
})
show_result(result2)
assert result2.status == "needs_confirmation"
assert result2.error_type == "exit_plan_mode_requested"
print(f"交互类型: {result2.metadata.get('interaction_type')}")

print("\n=== EnterPlanMode: 无原因 ===")
show_result(enter_plan_tool.run({}))

print("\n✅ EnterPlanMode & ExitPlanMode Tools 测试完成")

await test_agent_with_tool(registry, '这是一个复杂任务，请向我申请进入计划模式。')


=== EnterPlanMode: 请求进入计划模式 ===
状态: needs_confirmation
内容: 请求进入 plan 模式，等待调用方确认。
结构化数据: {
  "allowedActions": [
    "read_files",
    "search"
  ],
  "reason": "需求尚不明确，需要先做技术调研",
  "message": "请求进入 plan 模式，等待调用方确认。"
}
错误类型: enter_plan_mode_requested
交互类型: enter_plan_mode

=== ExitPlanMode: 请求退出计划模式 ===
状态: needs_confirmation
内容: 请求退出 plan 模式，等待调用方确认允许的执行权限。
结构化数据: {
  "allowedPrompts": [
    {
      "tool": "Bash",
      "prompt": "允许执行构建命令"
    }
  ],
  "message": "请求退出 plan 模式，等待调用方确认允许的执行权限。"
}
错误类型: exit_plan_mode_requested
交互类型: exit_plan_mode

=== EnterPlanMode: 无原因 ===
状态: needs_confirmation
内容: 请求进入 plan 模式，等待调用方确认。
结构化数据: {
  "allowedActions": [],
  "reason": "",
  "message": "请求进入 plan 模式，等待调用方确认。"
}
错误类型: enter_plan_mode_requested

✅ EnterPlanMode & ExitPlanMode Tools 测试完成

=== [Agent 调用测试] ===
提问: 这是一个复杂任务，请向我申请进入计划模式。
round 1

tool_calls:
EnterPlanMode : {'reason': '用户指示这是一个复杂任务，需要进入计划模式进行详细分析和方案设计。'}

interrupt:
请求进入 plan 模式，等待调用方确认。
Agent最终回复: 


In [15]:
# ==================== 20. EnterWorktree + ExitWorktree Tools ====================
from Tool.builtin import register_worktree_tools
from Tool.runtime import WorktreeManager

registry = ToolRegistry()

# WorktreeManager 需要 git 仓库根目录
try:
    repo_root = WorktreeManager.detect_repo_root(project_root)
    wt_manager = WorktreeManager(repo_root)
    enter_wt, exit_wt = register_worktree_tools(registry, worktree_manager=wt_manager)

    print("=== EnterWorktree: Schema 检查 ===")
    schema = enter_wt.get_openai_schema()
    print(f"工具名: {schema['function']['name']}")
    print(f"参数: {list(schema['function']['parameters'].get('properties', {}).keys())}")

    print("\n=== ExitWorktree: Schema 检查 ===")
    schema2 = exit_wt.get_openai_schema()
    print(f"工具名: {schema2['function']['name']}")
    print(f"参数: {list(schema2['function']['parameters'].get('properties', {}).keys())}")

    print("\n=== ExitWorktree: 无活动 worktree 时退出 ===")
    show_result(exit_wt.run({"action": "keep", "discard_changes": False}))

    # 注意：不执行 EnterWorktree 的实际创建，以避免修改 git 状态
    print("\n(跳过实际创建 worktree，避免修改 git 状态)")

except Exception as e:
    print(f"⚠️ Worktree 工具跳过测试: {e}")
    print("(项目不在 git 仓库中，或 git 不可用)")

print("\n✅ EnterWorktree & ExitWorktree Tools 测试完成")

await test_agent_with_tool(registry, '我要修改一个危险文件，请帮我开启一个隔离的 worktree。')


=== EnterWorktree: Schema 检查 ===
工具名: EnterWorktree
参数: ['name']

=== ExitWorktree: Schema 检查 ===
工具名: ExitWorktree
参数: ['action', 'discard_changes']

=== ExitWorktree: 无活动 worktree 时退出 ===
状态: error
内容: 退出 worktree 失败: 当前没有活动 worktree。
错误类型: worktree_exit_failed

(跳过实际创建 worktree，避免修改 git 状态)

✅ EnterWorktree & ExitWorktree Tools 测试完成

=== [Agent 调用测试] ===
提问: 我要修改一个危险文件，请帮我开启一个隔离的 worktree。
round 1

tool_calls:
EnterWorktree : {'name': 'dangerous-file-modification'}

round 2

content:
已为您开启并进入了名为 `dangerous-file-modification` 的隔离 worktree。您现在可以在这个安全的环境中进行危险文件的修改了。
final res:
已为您开启并进入了名为 `dangerous-file-modification` 的隔离 worktree。您现在可以在这个安全的环境中进行危险文件的修改了。
Agent最终回复: 已为您开启并进入了名为 `dangerous-file-modification` 的隔离 worktree。您现在可以在这个安全的环境中进行危险文件的修改了。


In [ ]:
# ==================== 21. Tool Schema 全量校验 ====================
from Tool.builtin import (
    register_filesystem_tools, register_file_write_tool, register_file_edit_tool,
    register_shell_tools, register_calculator_tool, register_web_fetch_tool,
    register_search_tool, register_todo_write_tool, register_config_tool,
    register_notebook_edit_tool, register_ask_user_question_tool,
    register_enter_plan_mode_tool, register_exit_plan_mode_tool,
)
from Tool.builtin import register_task_tools
from task import SQLiteTaskStore, TaskService

registry = ToolRegistry()
register_calculator_tool(registry)
register_filesystem_tools(registry, workspace_root=project_root)
register_file_write_tool(registry, workspace_root=project_root)
register_file_edit_tool(registry, workspace_root=project_root)
register_shell_tools(registry, workspace_root=project_root)
register_web_fetch_tool(registry)
register_search_tool(registry)
register_todo_write_tool(registry)
register_config_tool(registry)
register_notebook_edit_tool(registry, workspace_root=project_root)
register_ask_user_question_tool(registry)
register_enter_plan_mode_tool(registry)
register_exit_plan_mode_tool(registry)

db_path = os.path.join(SCRATCH_DIR, "schema_test.db")
service = TaskService(SQLiteTaskStore(db_path))
register_task_tools(registry, service=service)

print("=== 校验所有工具的 OpenAI Schema ===")
tools = registry.get_openai_tools()
all_passed = True

def check_schema(schema, path=""):
    global all_passed
    if not isinstance(schema, dict):
        return
    req = schema.get("required", [])
    props = list(schema.get("properties", {}).keys())
    if req and props:
        missing = [r for r in req if r not in props]
        if missing:
            print(f"  ❌ {path}: required={req} 但 properties 缺少 {missing}")
            all_passed = False
    for k, v in schema.get("properties", {}).items():
        check_schema(v, f"{path}.{k}")
    if "items" in schema and isinstance(schema["items"], dict):
        check_schema(schema["items"], f"{path}[items]")

for i, tool in enumerate(tools):
    name = tool["function"]["name"]
    params = tool["function"]["parameters"]
    required = params.get("required", [])
    properties = list(params.get("properties", {}).keys())
    missing = [r for r in required if r not in properties]
    status = "❌" if missing else "✅"
    print(f"[{i:>2}] {name:25s} {status}")
    if missing:
        print(f"     required 中缺少: {missing}")
        all_passed = False
    check_schema(params, name)

os.remove(db_path)

if all_passed:
    print(f"\n✅ 全部 {len(tools)} 个工具 Schema 校验通过！")
else:
    print("\n❌ 存在 Schema 问题，请检查上方输出")

In [16]:
# ==================== 清理测试目录 ====================
import shutil

if os.path.exists(SCRATCH_DIR):
    shutil.rmtree(SCRATCH_DIR)
    print(f"已清理测试目录: {SCRATCH_DIR}")
print("\n🎉 所有工具测试已完成！")

已清理测试目录: /home/wxd/LLM/EasyAgent/example/scratch/tool_test

🎉 所有工具测试已完成！
